# Activity 6: Evaluating RAG, and Judging With a Model

**Week 6 Day 4 | Four configurations, one eval set, and a number instead of an opinion**

**Estimated time:** 75 minutes
**Difficulty:** Intermediate
**Format:** Individual
**Prerequisites:** [Activity 5](./Activity_5_MLflow_Tour.ipynb) complete, same `.venv-mlflow` kernel

**Kernel reminder:** this notebook uses `.venv-mlflow`, not the root `.venv`. Same as Activity 5.

## The job

Today you built four retrievers and a RAG pipeline, and you decided which ones were good by **looking at two or three queries you picked yourself**. That is how every RAG project starts, and it is not evidence. You chose the queries after knowing what the corpus contained, you looked at a handful of results, and you formed an impression.

Impressions do not survive contact with a stakeholder asking "is it better than what we had?"

This notebook replaces the impression with a measurement:

1. Write down an **eval set** of questions with known correct answers, before measuring anything.
2. Measure **retrieval** with a cheap, objective metric that needs no LLM at all.
3. Measure **answer quality** with an LLM judge, and calibrate the judge before trusting it.
4. Run **four configurations** and log every one to MLflow, so the comparison is a table.

## What you will learn

- How to build an eval set that can actually change your mind
- Why retrieval quality and answer quality are separate measurements with separate failure modes
- How to test whether a judge is working, instead of assuming it is
- Why the corpus you evaluate on has to be harder than the corpus you demo on
- How to run an experiment grid and read the result

---
## Setup

In [ ]:
import os
import re
import json
import numpy as np
import mlflow
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine_similarity

load_dotenv()
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("week6-day4-rag-eval")

print("tracking uri:", mlflow.get_tracking_uri())

Same relative tracking URI as Activity 5, for the same reason: an absolute path containing a space silently sends your runs to a phantom `Week%206` folder.

---
# 1. A harder corpus, on purpose

Activities 3 and 4 used three papers. This notebook uses **all nine** in `pdfs/`.

That is not padding. A demo corpus and an eval corpus have different jobs. With three papers, "did retrieval find the right document" is a one-in-three guess, and both retrievers you are about to compare score near perfect, so the metric cannot move and the comparison teaches you nothing.

**A metric that cannot go down cannot tell you anything went wrong.** Adding six more papers of closely related NLP research means the wrong answers are now genuinely tempting, which is the only condition under which a measurement is informative.

In [ ]:
def extract_text(path):
    return " ".join(page.extract_text() for page in PdfReader(path).pages)


def chunk_text(text, chunk_size=800, overlap=100):
    stride = chunk_size - overlap
    return [text[i:i + chunk_size] for i in range(0, len(text), stride)]


PAPER_FILES = sorted(f for f in os.listdir("pdfs") if f.endswith(".pdf"))

CORPUS = []
for fname in PAPER_FILES:
    for i, chunk in enumerate(chunk_text(extract_text(f"pdfs/{fname}"))):
        CORPUS.append({"id": f"{fname}#{i}", "source": fname, "text": chunk})

print(f"{len(PAPER_FILES)} papers, {len(CORPUS)} chunks")
for f in PAPER_FILES:
    print(f"  {f:<22} {sum(1 for d in CORPUS if d['source'] == f):>3} chunks")

---
# 2. The eval set

An eval set is a list of questions where **you already know the right answer**. It is the most valuable artifact in this notebook and the one teams most often skip, because writing it is unglamorous and cannot be automated by the thing you are testing.

Three rules make one trustworthy:

1. **Write the questions before you look at any results.** Otherwise you will unconsciously write questions your current system happens to answer, and measure nothing but your own optimism.
2. **Record what the right answer is**, not just which question you asked. Without a reference you can only measure whether the system said *something*.
3. **Include the hard cases deliberately.** Activity 3 proved keyword search fails on paraphrases. An eval set with no paraphrases in it will report that keyword search is excellent.

This set has ten questions across three of the nine papers, deliberately split between two phrasing styles.

In [ ]:
BPE, W2V, GLOVE = "1508.07909v5.pdf", "1301.3781v3.pdf", "glove.pdf"

EVAL_SET = [
    {"question": "What are subword units used for in neural machine translation?",
     "expected_source": BPE, "style": "paper-vocabulary",
     "reference_answer": "They let the model encode rare and unknown words as sequences of smaller units, so it can translate words it never saw during training."},
    {"question": "What is byte pair encoding?",
     "expected_source": BPE, "style": "paper-vocabulary",
     "reference_answer": "A compression-derived algorithm that repeatedly merges the most frequent pair of symbols, used here to split words into subword units."},
    {"question": "How can a system translate a word it has never seen before?",
     "expected_source": BPE, "style": "paraphrase",
     "reference_answer": "By breaking the unknown word into smaller subword units it does know, and translating those, rather than emitting an unknown-word token."},
    {"question": "How do you handle words that are missing from the dictionary?",
     "expected_source": BPE, "style": "paraphrase",
     "reference_answer": "Segment them into subword units so the vocabulary stays open-ended, instead of replacing them with a single unknown token."},
    {"question": "What is the CBOW architecture?",
     "expected_source": W2V, "style": "paper-vocabulary",
     "reference_answer": "A model that predicts the current word from its surrounding context words, with the order of context words ignored."},
    {"question": "What is the skip-gram model?",
     "expected_source": W2V, "style": "paper-vocabulary",
     "reference_answer": "A model that uses the current word to predict the surrounding context words, which is the reverse of CBOW."},
    {"question": "How do you test whether word vectors capture meaning?",
     "expected_source": W2V, "style": "paraphrase",
     "reference_answer": "With analogy questions such as king minus man plus woman, checking whether the nearest vector is the expected word."},
    {"question": "What is the weighting function in the GloVe cost function?",
     "expected_source": GLOVE, "style": "paper-vocabulary",
     "reference_answer": "A function that damps the influence of very frequent co-occurrences, rising to a cap at x_max and using an exponent of 3/4."},
    {"question": "How does this method use word co-occurrence counts?",
     "expected_source": GLOVE, "style": "paraphrase",
     "reference_answer": "It trains on the nonzero entries of a global word-word co-occurrence matrix, fitting vectors so their dot product matches the log of the co-occurrence count."},
    {"question": "Which optimizer was used to train the model?",
     "expected_source": GLOVE, "style": "paraphrase",
     "reference_answer": "AdaGrad, applied by stochastically sampling the nonzero elements of the co-occurrence matrix."},
]

print(f"{len(EVAL_SET)} questions")
for style in ["paper-vocabulary", "paraphrase"]:
    print(f"  {style:<18} {sum(1 for e in EVAL_SET if e['style'] == style)}")

Note the `style` field. It is not used by any metric, it is there so you can slice the results afterwards and ask *where* a configuration failed, not just how often. A single average hides the exact pattern Activity 3 spent an hour demonstrating.

---
# 3. Two retrievers, held fair

To compare two things, change one thing. Both retrievers below take the same arguments and return the same shape, so the generation step downstream cannot tell them apart.

In [ ]:
texts = [doc["text"] for doc in CORPUS]

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(texts)


def embed_batch(items, batch_size=100):
    out = []
    for i in range(0, len(items), batch_size):
        r = client.embeddings.create(model="text-embedding-3-small", input=items[i:i + batch_size])
        out.extend(d.embedding for d in r.data)
    return out


corpus_embeddings = np.array(embed_batch(texts))
corpus_embeddings /= np.linalg.norm(corpus_embeddings, axis=1, keepdims=True)

print(f"tfidf matrix {tfidf_matrix.shape}, embeddings {corpus_embeddings.shape}")

In [ ]:
def retrieve_keyword(query, top_k=3):
    scores = sk_cosine_similarity(vectorizer.transform([query]), tfidf_matrix)[0]
    return [CORPUS[i] for i in np.argsort(-scores)[:top_k]]


def retrieve_embedding(query, top_k=3):
    qv = np.array(client.embeddings.create(model="text-embedding-3-small", input=query).data[0].embedding)
    qv /= np.linalg.norm(qv)
    return [CORPUS[i] for i in np.argsort(-(corpus_embeddings @ qv))[:top_k]]


RETRIEVERS = {"keyword": retrieve_keyword, "embedding": retrieve_embedding}

for name, fn in RETRIEVERS.items():
    print(name, "->", [d["source"] for d in fn(EVAL_SET[2]["question"])])

---
# 4. Metric one: did retrieval find the right document?

Start with the cheapest useful measurement. **Retrieval hit rate** asks a yes-or-no question per query: was the expected source document anywhere in the top `k`?

It costs nothing, needs no LLM, is perfectly objective, and it isolates the half of the pipeline that generation cannot fix. **If the right document was never retrieved, no prompt and no model will produce the right answer.** Measure this before you measure anything expensive.

In [ ]:
# TODO: Write retrieval_hit_rate(retriever, top_k).
#
# For every item in EVAL_SET:
#   - retrieve top_k docs for item["question"]
#   - it is a HIT if any returned doc's "source" equals item["expected_source"]
#
# Return the fraction of items that were hits, as a float between 0 and 1.
#
# Hint: any(d["source"] == item["expected_source"] for d in docs)
# Hint: sum(...) / len(EVAL_SET), and note that summing booleans works in
#       Python because True is 1.
#
# Expected: both retrievers should land somewhere between 0.5 and 1.0.
# If either one returns exactly 0.0 or exactly 1.0 at every top_k, stop and
# check your comparison, because a metric that never moves is broken.

def retrieval_hit_rate(retriever, top_k=3):
    pass


for name, fn in RETRIEVERS.items():
    for k in [1, 3]:
        print(f"{name:<10} top_k={k}  hit rate = {retrieval_hit_rate(fn, k):.0%}")

<details>
<summary>Still stuck? Hint: the loop</summary>

```python
def retrieval_hit_rate(retriever, top_k=3):
    hits = 0
    for item in EVAL_SET:
        docs = retriever(item["question"], top_k=top_k)
        hits += any(d["source"] == item["expected_source"] for d in docs)
    return hits / len(EVAL_SET)
```

`any(...)` returns a bool, and `hits += True` adds 1. If you prefer it explicit, write `if any(...): hits += 1`.

</details>

Read those four numbers before going on.

Embedding retrieval should beat keyword retrieval, and the gap should be wider at `top_k=1` than at `top_k=3`. That is not a coincidence: a larger `k` forgives a bad ranking, because the right document only has to appear *somewhere* in the list. A metric measured at `k=3` is more generous than the same metric at `k=1`, and quoting a hit rate without quoting the `k` is close to meaningless.

Now slice by phrasing style, which is where the real finding is.

In [ ]:
for name, fn in RETRIEVERS.items():
    for style in ["paper-vocabulary", "paraphrase"]:
        subset = [e for e in EVAL_SET if e["style"] == style]
        hits = sum(any(d["source"] == e["expected_source"] for d in fn(e["question"], 3)) for e in subset)
        print(f"{name:<10} {style:<18} {hits}/{len(subset)}")

This is the table worth keeping. An overall average would tell you embeddings are somewhat better; the split tells you *why*, and it should match what Activity 3 showed you directly.

If keyword retrieval holds up on paper-vocabulary questions and falls behind on paraphrases, you have independently reproduced Activity 3's finding as a measurement rather than an anecdote. That is the whole point of the exercise.

---
# 5. Metric two: was the answer any good?

Retrieval hit rate says the evidence was available. It says nothing about whether the model used it well. For that you need to compare a generated answer against your reference answer, and that comparison is harder than it looks.

Exact string matching fails immediately, for exactly the reason keyword search failed in Activity 3: two correct answers can share almost no words.

In [ ]:
reference = EVAL_SET[2]["reference_answer"]
candidate = "The model splits an unfamiliar word into smaller pieces it recognises and translates those pieces instead."

print("reference:", reference)
print("candidate:", candidate)
print()
print("exact match      :", reference.strip().lower() == candidate.strip().lower())
print("word overlap     :", len(set(reference.lower().split()) & set(candidate.lower().split())), "shared words")

That candidate is a correct answer. Exact match scores it zero, and the handful of shared words are mostly function words. Any metric built on literal overlap will punish a correct answer for using different vocabulary, which is precisely the failure you spent Activity 3 learning to recognise.

So you use a second model as a grader. This is **LLM-as-judge**, and it works because judging "do these two statements say the same thing" is an easier task than producing the answer in the first place.

In [ ]:
def rag_answer(question, retriever, top_k=3):
    docs = retriever(question, top_k=top_k)
    context = "\n\n".join(f"[{d['id']}] {d['text']}" for d in docs)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {"role": "system", "content": "Answer using only the provided excerpts. If they do not answer the question, say so."},
            {"role": "user", "content": f"Excerpts:\n{context}\n\nQuestion: {question}"},
        ],
    )
    return {"answer": response.choices[0].message.content, "sources": [d["source"] for d in docs]}

In [ ]:
JUDGE_PROMPT = '''You are grading whether a candidate answer is factually consistent with a reference answer.

Question: {question}
Reference answer: {reference}
Candidate answer: {candidate}

Score 1 if the candidate states the same core fact as the reference, even in different words.
Score 0 if it contradicts the reference, omits the core fact, or says the information is unavailable.

Respond with only a JSON object: {{"score": 0 or 1, "reason": "one short sentence"}}'''

print(JUDGE_PROMPT)

Three things about that prompt are deliberate.

**It asks for a binary score, not a 1 to 10 rating.** Models are unreliable at fine-grained scales, and the difference between a 6 and a 7 is noise you will then average and present as if it meant something. A coarse scale you can trust beats a fine one you cannot.

**It states the scoring rule explicitly**, including what earns a 0. "Grade this answer" without a rubric invites the model to invent its own standard, and it will invent a different one for different questions.

**It demands JSON.** You are parsing this in a loop, so the reply has to be machine-readable. The doubled braces `{{` and `}}` survive `.format()` as literal braces, which is how the prompt shows a JSON example without `.format` trying to substitute it.

In [ ]:
# TODO: Write judge_answer(question, candidate, reference).
#
#   1. Fill JUDGE_PROMPT with the three values using .format(...)
#   2. Call gpt-4o-mini with temperature=0 and that prompt as a user message
#   3. Parse the JSON reply and return the resulting dict
#
# Why temperature=0: a grader that gives different scores for identical
# input is not a grader. You cannot remove all judge noise, but you should
# not add any yourself.
#
# Hint: json.loads(response.choices[0].message.content)
# Hint: models sometimes wrap JSON in ```json fences. If json.loads raises,
#       strip them first. A tolerant version:
#           text = response.choices[0].message.content.strip()
#           text = re.sub(r"^```(?:json)?|```$", "", text, flags=re.M).strip()

def judge_answer(question, candidate, reference):
    pass

<details>
<summary>Still stuck? Hint: the whole function</summary>

```python
def judge_answer(question, candidate, reference):
    prompt = JUDGE_PROMPT.format(question=question, reference=reference, candidate=candidate)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[{"role": "user", "content": prompt}],
    )
    text = response.choices[0].message.content.strip()
    text = re.sub(r"^```(?:json)?|```$", "", text, flags=re.M).strip()
    return json.loads(text)
```

Note the doubled braces in `JUDGE_PROMPT`: `{{"score": ...}}`. They survive `.format()` as literal `{` and `}`, which is what lets the prompt show the model a JSON example without `.format` trying to substitute it.

</details>

## 5.1 Calibrate the judge before trusting it

Here is the step almost everyone skips. You are about to use this judge to make a decision, so first find out whether it can tell right from wrong at all.

Give it three answers where **you** already know the correct grade: one clearly right, one clearly wrong, and one that is right but worded completely differently.

In [ ]:
q = EVAL_SET[2]["question"]
ref = EVAL_SET[2]["reference_answer"]

calibration = [
    ("should score 1, correct and similar wording",
     "It splits the unknown word into subword units it has seen and translates those."),
    ("should score 1, correct but very different wording",
     "Rather than giving up on an unfamiliar term, the system chops it into familiar fragments and works from those."),
    ("should score 0, confidently wrong",
     "It looks the word up in a bilingual dictionary that ships with the model."),
]

for expectation, candidate in calibration:
    verdict = judge_answer(q, candidate, ref)
    print(f"{expectation}\n   -> score={verdict['score']}  {verdict['reason']}\n")

Check each verdict against the expectation on its line. A judge that scores the wrong answer as 1, or either correct answer as 0, is not usable, and you would fix the prompt now rather than discover it after logging four runs.

Judges are not neutral instruments, and the known problems are worth naming:

- **They are noisy.** Same input, different runs, occasionally different scores. `temperature=0` reduces this but does not eliminate it.
- **They share blind spots with the model they grade.** Here the judge and the candidate are the *same model*, so a confident misconception is one the judge may well agree with.
- **They tend to reward fluency.** A well-written wrong answer scores better than it should.

None of that makes the judge useless. It makes it a **relative** instrument. "Configuration A scored higher than configuration B on the same eval set with the same judge" is a defensible claim. "This pipeline is 87 percent accurate" is not.

---
# 6. The experiment grid

Two retrievers, two values of `top_k`, four runs. Each run logs its configuration as params and its results as metrics, which is exactly the shape Activity 5 set up.

In [ ]:
# TODO: Write run_eval(retriever_name, top_k) to evaluate one configuration
# and log it as a single MLflow run.
#
# Inside `with mlflow.start_run(run_name=f"{retriever_name}_k{top_k}"):`
#
#   1. Log params: "retriever" (the name) and "top_k".
#   2. For each item in EVAL_SET:
#        - result = rag_answer(item["question"], retriever, top_k=top_k)
#        - hit = item["expected_source"] in result["sources"]
#        - verdict = judge_answer(item["question"], result["answer"],
#                                 item["reference_answer"])
#        - collect a row with question, style, hit, verdict["score"],
#          the answer, and verdict["reason"]
#   3. Log two metrics:
#        "retrieval_hit_rate" -> mean of hit
#        "mean_judge_score"   -> mean of the judge scores
#   4. Save the per-question rows as an artifact:
#        mlflow.log_dict({"results": rows}, "per_question_results.json")
#   5. Return rows
#
# Log the per-question artifact, not just the averages. When a
# configuration scores badly, the average tells you THAT it failed and the
# artifact tells you WHICH questions failed. You will want the second one.
#
# This makes 20 API calls per configuration and takes about a minute.

def run_eval(retriever_name, top_k=3):
    pass

<details>
<summary>Still stuck? Hint 1: the per-question loop</summary>

```python
for item in EVAL_SET:
    result = rag_answer(item["question"], retriever, top_k=top_k)
    hit = item["expected_source"] in result["sources"]
    verdict = judge_answer(item["question"], result["answer"], item["reference_answer"])
    rows.append({"question": item["question"], "style": item["style"],
                 "retrieval_hit": bool(hit), "judge_score": verdict["score"],
                 "answer": result["answer"], "reason": verdict["reason"]})
```

`result["sources"]` is a list of filenames, so `in` does the membership test for you.

</details>

<details>
<summary>Still stuck? Hint 2: the metrics and the artifact</summary>

```python
mlflow.log_metric("retrieval_hit_rate", sum(r["retrieval_hit"] for r in rows) / len(rows))
mlflow.log_metric("mean_judge_score", sum(r["judge_score"] for r in rows) / len(rows))
mlflow.log_dict({"results": rows}, "per_question_results.json")
```

`log_dict` needs something JSON-serialisable. If it raises, you probably stored a numpy bool rather than a Python one, which is why the hint above wraps it in `bool(...)`.

</details>

In [ ]:
all_rows = {}
for retriever_name in ["keyword", "embedding"]:
    for k in [1, 3]:
        all_rows[(retriever_name, k)] = run_eval(retriever_name, top_k=k)
        print(f"done: {retriever_name} k={k}")

---
# 7. Read the result

In [ ]:
runs = mlflow.search_runs(experiment_names=["week6-day4-rag-eval"])

table = runs[["params.retriever", "params.top_k", "metrics.retrieval_hit_rate", "metrics.mean_judge_score"]]
table.sort_values(["params.retriever", "params.top_k"]).reset_index(drop=True)

Four rows, two numbers each, one variable changing at a time. Read them in this order:

**Compare retrievers at the same `top_k`.** That isolates the retrieval method, and it should reproduce the section 4 result.

**Compare `top_k` within one retriever.** Higher `k` should raise the hit rate, because more chances to include the right document. Watch whether `mean_judge_score` rises with it, and be ready for it not to: more context also means more irrelevant text competing for the model's attention, which is the trade-off Activity 4 priced in tokens.

**Notice how much lower `mean_judge_score` is than `retrieval_hit_rate`.** Do not skip past that. This pipeline is finding the right paper most of the time and still producing answers the judge rejects, on a majority of questions at `top_k=1`.

That is the honest state of a naive RAG pipeline pointed at nine dense research papers with 800-character fixed-size chunks, and it is worth seeing. A tutorial that reports 95 percent on ten easy questions has taught you how to build a demo. The gap you are looking at is what the rest of the work in this field is actually about.

**Compare the two metrics against each other.** This is the interesting one.

In [ ]:
for (name, k), rows in all_rows.items():
    retrieved_but_wrong = [r for r in rows if r["retrieval_hit"] and r["judge_score"] == 0]
    missed_but_right = [r for r in rows if not r["retrieval_hit"] and r["judge_score"] == 1]
    print(f"{name:<10} k={k}  found the doc but answered badly: {len(retrieved_but_wrong)}"
          f"   |  missed the doc but answered well: {len(missed_but_right)}")

Those two counts point at different problems, and the first one is probably large.

**Found the document, answered badly.** Resist the obvious reading. It is tempting to call this a pure generation failure, but look at what `retrieval_hit_rate` actually measures here: it checks whether the right **paper** appeared, not whether the passage that answers the question appeared. These papers are around 80 chunks each. Retrieving the correct paper and still missing the one paragraph that contains the answer is easy, especially at `top_k=1`.

So this count mixes two causes:

- the answering passage really was in the context and the model mishandled it, which is a generation problem, or
- only the right *paper* was retrieved and the right *passage* was not, which is still a retrieval problem that your metric is too coarse to see.

**A loose metric produces a confident misdiagnosis.** The fix is to tighten it, by labelling the expected chunk id rather than the source file, which is more work to author and is exactly why the first version of an eval set is rarely the last.

**Missed the document, answered well** is rarer and more alarming: the model produced an answer the judge liked *without* the right source. Sometimes a sibling paper genuinely covers the same ground. Sometimes the model answered from training knowledge and ignored your context entirely, which means your RAG system is not really doing RAG, and the judge cannot tell because it only ever sees the answer.

Print both categories and read the actual text, rather than trusting either count.

In [ ]:
suspicious = [(name, k, r) for (name, k), rows in all_rows.items()
              for r in rows if not r["retrieval_hit"] and r["judge_score"] == 1]

if not suspicious:
    print("No 'missed the document but answered well' cases in this run.")
    print("That is a good sign: it suggests the model is not quietly answering from memory.")
else:
    for name, k, r in suspicious:
        print(f"--- {name} k={k} ---")
        print("Q:", r["question"])
        print("A:", r["answer"][:220])
        print("judge:", r["reason"], "\n")

print("\n=== retrieved the right paper, still scored 0 ===")
for (name, k), rows in all_rows.items():
    for r in rows:
        if r["retrieval_hit"] and r["judge_score"] == 0:
            print(f"[{name} k={k}] {r['question']}")
            print(f"    judge: {r['reason']}")

---
# 8. What you can and cannot claim

You now have four measured configurations. Be precise about what that buys you.

**Supported by this evidence:**

- One configuration scored higher than another, on this eval set, with this judge.
- Retrieval quality and answer quality moved differently, and you can point at which questions caused it.
- The paraphrase questions behaved differently from the paper-vocabulary ones.

**Not supported:**

- Any absolute quality claim. Ten questions is a small sample, and one flipped question moves a rate by ten points, which is larger than most of the gaps you just measured.
- That this generalises to your users' real questions. You wrote these questions, and you are not your user.
- That the judge is right. It is a noisy instrument grading its own model family.

The honest summary is comparative and hedged, and it is still far more than the impression you started the notebook with. **The point of an eval set is not to produce a number you trust absolutely. It is to make a change measurable, so that "this made it better" stops being a matter of opinion.**

Everything you would do next is now cheap, because the harness exists: add questions, swap in Activity 3's BM25 or hybrid retriever, change the chunk size, try a stronger judge. Each one is another row in the same table.

---
# Your Turn

Work in your own copy under `student-work/week6/day4/`.

1. **Add your hybrid retriever.** Bring `bm25_search` and the RRF `hybrid_search` from Activity 3 into `RETRIEVERS`, then run the grid again. Activity 3 argued hybrid should beat both. Does the measurement agree? If it does not, say what you would check first.

2. **Break the eval set on purpose.** Delete the five paraphrase questions and re-run just `keyword_k3`. Watch its scores rise. Write two sentences on what this proves about eval sets you did not write yourself.

3. **Test judge stability.** Pick one question and call `judge_answer` on the same candidate five times. Do you get five identical scores? Whatever you find, decide whether it changes how much you trust a ten-point gap between two configurations.

4. **Find the ceiling.** Add `top_k=10` to the grid. Does `retrieval_hit_rate` keep climbing? Does `mean_judge_score` follow it, or peak and fall? Explain the shape you get.

**Stretch goal:** the judge currently returns 0 or 1. Change it to score groundedness separately from correctness: is the answer supported *by the retrieved excerpts*, regardless of whether it is true? Log both metrics. A high correctness with low groundedness score is the signature of a model answering from memory, and it is the number that would actually catch the "missed the doc but answered well" case in section 7.

## What you did

- Built an eval set with reference answers and a phrasing-style label, before measuring anything.
- Used a nine-paper corpus, because a three-paper corpus made the metric unable to move.
- Measured retrieval with a cheap objective metric that needs no LLM, and sliced it by question style.
- Showed why exact match cannot grade a correct answer that uses different words.
- Built an LLM judge and calibrated it on known-right and known-wrong answers before trusting it.
- Ran a four-configuration grid and logged every run to MLflow with a per-question artifact.
- Separated retrieval failures from generation failures, and found the case where a good score hides a broken pipeline.
- Wrote down what the evidence does and does not support.

**Next:** [Activity 7](./Activity_7_From_Notebook_to_Pipeline.ipynb) takes the classification work from Activity 2 and turns it into something you could actually run over 400,000 rows: schema-constrained output, a cost estimate up front, bounded concurrency, and a resumable job.